# 🧬 BiomeX — Pipeline Complet : Microbiome → Prédictions Cliniques

> **Version 4.0 — Février 2026**  
> Auteurs : Abdou Bâ (CTO) · Mouhammadou Dia (CEO)  
> Contact : abdou.ba@biomex.ai · www.biomex.ai

---

## 📋 Table des matières

1. [Installation & Imports](#1)
2. [Chargement et simulation des données](#2)
3. [Pipeline bioinformatique (FastQC → HUMAnN3)](#3)
4. [Construction de la matrice globale](#4)
5. [Réduction de dimension](#5)
6. [Calcul des scores biologiques](#6)
7. [Modélisation prédictive supervisée](#7)
8. [Prédiction pour un nouvel individu](#8)
9. [Validation et interprétabilité (SHAP)](#9)
10. [Génération du rapport patient](#10)

---
⚠️ **Note** : Ce notebook utilise des **données simulées** pour la démonstration.  
En production, remplacer `simulate_*()` par les vraies sorties des outils bioinformatiques (Kraken2, HUMAnN3, etc.)

---
## 1. Installation & Imports <a id='1'></a>

In [ ]:
# ── Installation des dépendances ──────────────────────────────────────
# Décommenter si nécessaire
# !pip install numpy pandas scikit-learn xgboost shap matplotlib seaborn
# !pip install torch torchvision umap-learn scipy statsmodels
# !pip install reportlab fpdf2 ipywidgets tqdm

In [ ]:
# ── Imports standard ─────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import braycurtis
import warnings
warnings.filterwarnings('ignore')

# ── Machine Learning ─────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, mean_squared_error, f1_score
)
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel
import xgboost as xgb
import shap

# ── Deep Learning ─────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ── Config visuelle ───────────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# Palette BiomeX
BIOMEX_COLORS = {
    'blue':   '#0066CC',
    'green':  '#009944',
    'navy':   '#003366',
    'orange': '#E07B00',
    'teal':   '#007B8A',
    'purple': '#6B3FA0',
    'light':  '#D6E8FA',
}

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("✅ Environnement BiomeX prêt")
print(f"   NumPy  : {np.__version__}")
print(f"   Pandas : {pd.__version__}")
print(f"   Sklearn: {__import__('sklearn').__version__}")
print(f"   XGBoost: {xgb.__version__}")
print(f"   PyTorch: {torch.__version__}")
print(f"   SHAP   : {shap.__version__}")


---
## 2. Chargement des données (réelles + simulation) <a id='2'></a>

Ce notebook tente d'abord de charger les données de `Data/Data_proccesing`.
Si les fichiers ne sont pas disponibles, il bascule automatiquement en mode simulation.


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# CONFIG GLOBALE
# ══════════════════════════════════════════════════════════════════════
N_PATIENTS        = 500   # Cohorte pilote BiomeX Dakar (mode simulation)
N_BACTERIA        = 200   # Espèces bactériennes détectées
N_FUNCTIONS       = 300   # Voies fonctionnelles HUMAnN3 (KEGG, MetaCyc)
N_DIET_FEATURES   = 50    # Variables nutritionnelles
N_CLINICAL        = 30    # Variables cliniques

# Active l'utilisation des données Data/Data_proccesing quand disponibles
USE_PROCESSED_DATA = True
DATA_PROCESSING_CANDIDATES = [
    '../Data/Data_proccesing',
    'Data/Data_proccesing',
]

BACTERIA_NAMES = (
    # Phylum Firmicutes
    ['Lactobacillus_rhamnosus', 'Lactobacillus_acidophilus', 'Lactobacillus_fermentum',
     'Bifidobacterium_longum', 'Bifidobacterium_adolescentis', 'Faecalibacterium_prausnitzii',
     'Roseburia_intestinalis', 'Ruminococcus_gnavus', 'Blautia_obeum', 'Clostridium_butyricum']
    +
    # Phylum Bacteroidetes
    ['Bacteroides_fragilis', 'Bacteroides_thetaiotaomicron', 'Prevotella_copri',
     'Prevotella_stercorea', 'Alistipes_putredinis', 'Parabacteroides_distasonis']
    +
    # Phylum Proteobacteria (dont pathobiontes)
    ['Escherichia_coli', 'Klebsiella_pneumoniae', 'Helicobacter_pylori',
     'Enterobacter_cloacae', 'Citrobacter_freundii']
    +
    # Bactéries spécifiques Afrique de l'Ouest
    ['Akkermansia_muciniphila', 'Christensenella_minuta', 'Treponema_succinifaciens',
     'Spirochaeta_africana', 'Prevotella_africana']
    +
    [f'Bacterium_sp_{i}' for i in range(1, N_BACTERIA - 30)]
)
BACTERIA_NAMES = BACTERIA_NAMES[:N_BACTERIA]

# Pathobiontes connus (pour score de dysbiose)
PATHOBIONTES = [
    'Escherichia_coli', 'Klebsiella_pneumoniae', 'Helicobacter_pylori',
    'Enterobacter_cloacae', 'Citrobacter_freundii', 'Ruminococcus_gnavus'
]

# Bactéries productrices de butyrate (SCFA)
BUTYRATE_PRODUCERS = [
    'Faecalibacterium_prausnitzii', 'Roseburia_intestinalis',
    'Clostridium_butyricum', 'Butyrivibrio_fibrisolvens'
]

print(f"⚙️  Configuration : {N_PATIENTS} patients · {N_BACTERIA} espèces · {N_FUNCTIONS} fonctions")
print(f"   Mode data processing : {'activé' if USE_PROCESSED_DATA else 'désactivé'}")


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# CHARGEMENT DES DONNÉES (Data Processing) + FALLBACK SIMULATION
# ══════════════════════════════════════════════════════════════════════

import os


def resolve_data_processing_root(candidates):
    """Retourne le premier dossier Data_proccesing valide trouvé."""
    for root in candidates:
        meta_path = os.path.join(root, 'Viome_microbiome_metadata.csv')
        species_path = os.path.join(root, 'viome-abundance', 'abundance', 'Viome_species_readcount_40samples.csv')
        func_path = os.path.join(root, 'viome-abundance', 'abundance', 'Viome_function_KO_readcount_40samples.csv')
        if os.path.exists(meta_path) and os.path.exists(species_path) and os.path.exists(func_path):
            return root
    return None


def load_processed_viome_data(data_root, n_bacteria=200, n_functions=300):
    """Charge et pivote les sorties de data processing Viome."""
    species_path = os.path.join(data_root, 'viome-abundance', 'abundance', 'Viome_species_readcount_40samples.csv')
    func_path = os.path.join(data_root, 'viome-abundance', 'abundance', 'Viome_function_KO_readcount_40samples.csv')
    meta_path = os.path.join(data_root, 'Viome_microbiome_metadata.csv')

    df_species = pd.read_csv(species_path)
    df_func = pd.read_csv(func_path)
    df_meta = pd.read_csv(meta_path)

    for df in [df_species, df_func, df_meta]:
        df.columns = [c.strip().strip('"') for c in df.columns]

    # Taxonomie (species readcount)
    df_species['sample_name'] = df_species['sample_name'].astype(str).str.strip().str.strip('"')
    df_species['species_name'] = (
        df_species['species_name']
        .astype(str)
        .str.strip()
        .str.strip('"')
        .str.replace(' ', '_', regex=False)
    )
    df_species['species_readcount'] = pd.to_numeric(df_species['species_readcount'], errors='coerce').fillna(0)

    X_taxa_full = df_species.pivot_table(
        index='sample_name',
        columns='species_name',
        values='species_readcount',
        aggfunc='sum',
        fill_value=0,
    )
    top_species = X_taxa_full.sum(axis=0).sort_values(ascending=False).head(n_bacteria).index
    X_taxa = X_taxa_full.loc[:, top_species]

    # Fonctionnel (KO readcount)
    df_func['sample_name'] = df_func['sample_name'].astype(str).str.strip().str.strip('"')
    df_func['function_name'] = df_func['function_name'].astype(str).str.strip().str.strip('"')
    df_func['readcount'] = pd.to_numeric(df_func['readcount'], errors='coerce').fillna(0)

    X_func_full = df_func.pivot_table(
        index='sample_name',
        columns='function_name',
        values='readcount',
        aggfunc='sum',
        fill_value=0,
    )
    top_functions = X_func_full.sum(axis=0).sort_values(ascending=False).head(n_functions).index
    X_func = X_func_full.loc[:, top_functions]

    # Metadata (long format par échantillon)
    df_meta = df_meta.rename(columns={'symptome_T2': 'symptom_T2'})
    rows = []
    for _, row in df_meta.iterrows():
        for tp in ['T1', 'T2']:
            sample_col = f'sample_name_{tp}'
            symptom_col = f'symptom_{tp}'
            sample_name = row.get(sample_col)
            if pd.notna(sample_name):
                rows.append({
                    'sample_name': str(sample_name).strip(),
                    'subject_id': row.get('subject_id'),
                    'timepoint': tp,
                    'age': row.get('age'),
                    'gender': row.get('gender'),
                    'bmi': row.get('bmi'),
                    'symptom_level': row.get(symptom_col),
                })

    df_meta_long = pd.DataFrame(rows)
    if not df_meta_long.empty:
        df_meta_long = (
            df_meta_long
            .dropna(subset=['sample_name'])
            .drop_duplicates(subset=['sample_name'])
            .set_index('sample_name')
        )

    return X_taxa, X_func, df_meta_long


def simulate_microbiome_data(n_patients, n_bacteria, bacteria_names, seed=42):
    """
    Simule une matrice d'abondances microbiomiques.
    En production : lire la sortie Kraken2/QIIME2 (format BIOM ou TSV).
    """
    rng = np.random.RandomState(seed)

    # Profil de base : distribution Dirichlet (réaliste pour microbiome)
    alpha = rng.exponential(0.5, size=n_bacteria) + 0.01
    raw   = rng.dirichlet(alpha, size=n_patients)  # abondances relatives

    # Ajouter variabilité inter-individuelle
    noise = rng.normal(0, 0.005, size=raw.shape)
    raw   = np.clip(raw + noise, 0, None)
    raw   = raw / raw.sum(axis=1, keepdims=True)  # renormaliser

    # Multiplier par profondeur de séquençage (reads counts)
    depths = rng.randint(50_000, 200_000, size=n_patients)
    counts = (raw * depths[:, np.newaxis]).astype(int)

    df = pd.DataFrame(counts, columns=bacteria_names)
    df.index = [f'Patient_{i:04d}' for i in range(n_patients)]
    return df


def simulate_functional_data(n_patients, n_functions, seed=42):
    """
    Simule une matrice de fonctions biologiques (output HUMAnN3).
    En production : lire les fichiers *_genefamilies.tsv de HUMAnN3.
    """
    rng = np.random.RandomState(seed)
    func_names = (
        [f'KEGG_K{i:05d}' for i in range(n_functions // 3)] +
        [f'MetaCyc_PWY_{i}' for i in range(n_functions // 3)] +
        [f'GO_term_{i}' for i in range(n_functions - 2 * (n_functions // 3))]
    )
    data = rng.lognormal(mean=2.0, sigma=1.5, size=(n_patients, n_functions))
    data = np.clip(data, 0, None)
    df   = pd.DataFrame(data, columns=func_names)
    df.index = [f'Patient_{i:04d}' for i in range(n_patients)]
    return df


def simulate_diet_data(n_patients, n_features=50, seed=42):
    """Simule les données alimentaires (questionnaire FFQ)."""
    rng = np.random.RandomState(seed)
    cols = (
        ['fibers_g', 'proteins_g', 'carbs_g', 'fats_g', 'calories_kcal',
         'sugar_g', 'sodium_mg', 'iron_mg', 'zinc_mg', 'vitD_ug',
         'millet_freq', 'sorghum_freq', 'niebe_freq', 'baobab_freq',
         'vegetables_portions', 'fruits_portions', 'meat_portions',
         'fish_portions', 'dairy_portions', 'processed_food_freq']
        + [f'food_var_{i}' for i in range(n_features - 20)]
    )
    data = rng.exponential(scale=50, size=(n_patients, n_features))
    df   = pd.DataFrame(data, columns=cols[:n_features])
    df.index = [f'Patient_{i:04d}' for i in range(n_patients)]
    return df


def simulate_clinical_data(n_patients, seed=42):
    """Simule les données cliniques et métadonnées."""
    rng = np.random.RandomState(seed)
    df  = pd.DataFrame(index=[f'Patient_{i:04d}' for i in range(n_patients)])

    # Biomarqueurs
    df['age']           = rng.randint(18, 75, n_patients)
    df['sexe']          = rng.choice([0, 1], n_patients)          # 0=F, 1=M
    df['IMC']           = rng.normal(26, 5, n_patients).clip(15, 50)
    df['glycemie_mmol'] = rng.normal(5.5, 1.2, n_patients).clip(3, 15)
    df['HbA1c_pct']     = rng.normal(5.8, 0.9, n_patients).clip(4, 12)
    df['CRP_mg_L']      = rng.exponential(3, n_patients).clip(0, 50)
    df['tension_sys']   = rng.normal(125, 18, n_patients).clip(90, 200)
    df['tension_dia']   = rng.normal(80, 12, n_patients).clip(60, 120)
    df['cholesterol']   = rng.normal(4.8, 1.2, n_patients).clip(2, 10)
    df['triglycerides'] = rng.exponential(1.5, n_patients).clip(0.3, 10)

    # Symptômes digestifs (0-3)
    df['ballonnements']     = rng.randint(0, 4, n_patients)
    df['transit_score']     = rng.randint(0, 4, n_patients)
    df['douleurs_abdomen']  = rng.randint(0, 4, n_patients)
    df['fatigue_score']     = rng.randint(0, 4, n_patients)

    # Métadonnées
    df['ethnie'] = rng.choice(['Wolof', 'Sérère', 'Peul', 'Diola', 'Mandingue'], n_patients)
    df['zone']   = rng.choice(['Dakar_urbain', 'Dakar_banlieue', 'Zone_rurale'], n_patients,
                               p=[0.5, 0.35, 0.15])
    df['antibiotiques_3mois'] = rng.choice([0, 1], n_patients, p=[0.75, 0.25])

    # Labels cliniques (cibles)
    df['diabete_risque']      = ((df['glycemie_mmol'] > 7.0) | (df['HbA1c_pct'] > 6.5)).astype(int)
    df['syndrome_metabolique']= ((df['IMC'] > 30) & (df['glycemie_mmol'] > 6.1)).astype(int)
    df['inflammation_elevee'] = (df['CRP_mg_L'] > 10).astype(int)

    return df


def build_clinical_from_metadata(df_meta_aligned: pd.DataFrame, seed=42):
    """Construit un tableau clinique enrichi à partir du metadata + variables dérivées."""
    rng = np.random.RandomState(seed)
    idx = df_meta_aligned.index
    n = len(idx)

    df = pd.DataFrame(index=idx)

    age = pd.to_numeric(df_meta_aligned.get('age'), errors='coerce')
    bmi = pd.to_numeric(df_meta_aligned.get('bmi'), errors='coerce')

    df['age'] = age.fillna(pd.Series(rng.randint(18, 75, n), index=idx))

    gender = df_meta_aligned.get('gender', pd.Series(index=idx, dtype='object')).astype(str).str.upper()
    sexe_map = {'F': 0, 'M': 1}
    df['sexe'] = gender.map(sexe_map)
    df['sexe'] = df['sexe'].fillna(pd.Series(rng.choice([0, 1], n), index=idx)).astype(int)

    df['IMC'] = bmi.fillna(pd.Series(rng.normal(26, 4, n), index=idx)).clip(15, 50)

    # Variables cliniques dérivées (cohérentes avec âge + IMC)
    df['glycemie_mmol'] = (4.3 + 0.015 * (df['age'] - 35) + 0.055 * (df['IMC'] - 22)
                           + rng.normal(0, 0.7, n)).clip(3, 15)
    df['HbA1c_pct'] = (4.7 + 0.22 * (df['glycemie_mmol'] - 4.5)
                       + rng.normal(0, 0.35, n)).clip(4, 12)
    df['CRP_mg_L'] = (0.8 + 0.22 * (df['IMC'] - 20).clip(lower=0)
                      + rng.exponential(1.8, n)).clip(0, 50)
    df['tension_sys'] = (105 + 0.8 * (df['age'] - 18)
                         + 0.5 * (df['IMC'] - 20)
                         + rng.normal(0, 7, n)).clip(90, 200)
    df['tension_dia'] = (65 + 0.35 * (df['age'] - 18)
                         + 0.35 * (df['IMC'] - 20)
                         + rng.normal(0, 5, n)).clip(60, 120)
    df['cholesterol'] = (3.8 + 0.02 * (df['age'] - 18)
                         + 0.05 * (df['IMC'] - 22)
                         + rng.normal(0, 0.5, n)).clip(2, 10)
    df['triglycerides'] = (0.9 + 0.06 * (df['IMC'] - 22)
                           + rng.normal(0, 0.35, n)).clip(0.3, 10)

    # Symptômes digestifs dérivés du metadata
    symptom_map = {
        'none': 0,
        'mild': 1,
        'moderate': 2,
        'severe': 3,
    }
    symptoms_raw = df_meta_aligned.get('symptom_level', pd.Series(index=idx, dtype='object'))
    sev = (
        symptoms_raw
        .fillna('moderate')
        .astype(str)
        .str.strip()
        .str.lower()
        .map(symptom_map)
    )
    sev = sev.fillna(pd.Series(rng.randint(0, 4, n), index=idx)).astype(int)

    df['ballonnements'] = sev
    df['transit_score'] = (sev + rng.randint(-1, 2, n)).clip(0, 3).astype(int)
    df['douleurs_abdomen'] = (sev + rng.randint(-1, 2, n)).clip(0, 3).astype(int)
    df['fatigue_score'] = (sev + (df['CRP_mg_L'] > 8).astype(int)).clip(0, 3).astype(int)

    # Métadonnées (si absentes: valeurs synthétiques)
    df['ethnie'] = rng.choice(['Wolof', 'Sérère', 'Peul', 'Diola', 'Mandingue'], n)
    df['zone'] = rng.choice(['Dakar_urbain', 'Dakar_banlieue', 'Zone_rurale'], n,
                            p=[0.5, 0.35, 0.15])
    df['antibiotiques_3mois'] = rng.choice([0, 1], n, p=[0.75, 0.25])

    # Labels cliniques
    df['diabete_risque'] = ((df['glycemie_mmol'] > 7.0) | (df['HbA1c_pct'] > 6.5)).astype(int)
    df['syndrome_metabolique'] = ((df['IMC'] > 30) & (df['glycemie_mmol'] > 6.1)).astype(int)
    df['inflammation_elevee'] = (df['CRP_mg_L'] > 10).astype(int)

    return df


# ── Chargement prioritaire des données data processing ───────────────
DATA_SOURCE = 'simulation'
root = resolve_data_processing_root(DATA_PROCESSING_CANDIDATES) if USE_PROCESSED_DATA else None

if USE_PROCESSED_DATA and root is not None:
    try:
        print(f"📂 Chargement des données data processing : {root}")
        X_taxa_real, X_func_real, df_meta_real = load_processed_viome_data(
            root,
            n_bacteria=N_BACTERIA,
            n_functions=N_FUNCTIONS,
        )

        common_idx = X_taxa_real.index.intersection(X_func_real.index)
        if len(common_idx) < 10:
            raise ValueError(f"Pas assez d'échantillons communs ({len(common_idx)})")

        X_taxa = X_taxa_real.loc[common_idx].copy()
        X_func = X_func_real.loc[common_idx].copy()

        # Mise à jour de la configuration effective
        BACTERIA_NAMES = X_taxa.columns.tolist()
        N_PATIENTS = len(common_idx)
        N_BACTERIA = X_taxa.shape[1]
        N_FUNCTIONS = X_func.shape[1]

        X_diet = simulate_diet_data(N_PATIENTS, N_DIET_FEATURES, seed=SEED)
        X_diet.index = common_idx

        if isinstance(df_meta_real, pd.DataFrame) and not df_meta_real.empty:
            df_clin = build_clinical_from_metadata(df_meta_real.reindex(common_idx), seed=SEED)
        else:
            df_clin = simulate_clinical_data(N_PATIENTS, seed=SEED)
            df_clin.index = common_idx

        DATA_SOURCE = 'data_processing + simulation clinique'

    except Exception as exc:
        print(f"⚠️ Chargement data processing échoué ({exc}) -> fallback simulation")

if DATA_SOURCE == 'simulation':
    print("🔬 Génération des données simulées...")
    X_taxa  = simulate_microbiome_data(N_PATIENTS, N_BACTERIA, BACTERIA_NAMES)
    X_func  = simulate_functional_data(N_PATIENTS, N_FUNCTIONS)
    X_diet  = simulate_diet_data(N_PATIENTS)
    df_clin = simulate_clinical_data(N_PATIENTS)

X_symp  = df_clin.drop(columns=['ethnie', 'zone', 'diabete_risque',
                                'syndrome_metabolique', 'inflammation_elevee'])
Y = df_clin[['diabete_risque', 'syndrome_metabolique', 'inflammation_elevee']]

print(f"🧭 Source des données : {DATA_SOURCE}")
print(f"  ✅ Taxonomie    : {X_taxa.shape}  (patients × espèces)")
print(f"  ✅ Fonctionnel  : {X_func.shape}  (patients × fonctions)")
print(f"  ✅ Alimentaire  : {X_diet.shape}  (patients × variables)")
print(f"  ✅ Clinique     : {X_symp.shape}  (patients × biomarqueurs)")
print(f"  ✅ Labels       : {Y.shape}       (patients × cibles)")


In [ ]:
# ── Exploration rapide des données ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('BiomeX — Exploration des données cliniques', fontsize=16,
             fontweight='bold', color=BIOMEX_COLORS['navy'])

axes[0].hist(df_clin['glycemie_mmol'], bins=30,
             color=BIOMEX_COLORS['blue'], alpha=0.8, edgecolor='white')
axes[0].axvline(7.0, color='red', ls='--', label='Seuil diabète')
axes[0].set_title('Distribution Glycémie (mmol/L)')
axes[0].set_xlabel('Glycémie (mmol/L)')
axes[0].legend()

axes[1].hist(df_clin['IMC'], bins=30,
             color=BIOMEX_COLORS['green'], alpha=0.8, edgecolor='white')
axes[1].axvline(25, color='orange', ls='--', label='Surpoids')
axes[1].axvline(30, color='red',    ls='--', label='Obésité')
axes[1].set_title('Distribution IMC')
axes[1].set_xlabel('IMC (kg/m²)')
axes[1].legend()

prev = Y.mean() * 100
axes[2].bar(prev.index, prev.values,
            color=[BIOMEX_COLORS['blue'], BIOMEX_COLORS['orange'], BIOMEX_COLORS['purple']],
            edgecolor='white')
axes[2].set_title('Prévalence des pathologies (%)')
axes[2].set_ylabel('%')
for i, v in enumerate(prev.values):
    axes[2].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('01_exploration_donnees.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Sauvegardé : 01_exploration_donnees.png")

---
## 3. Pipeline Bioinformatique <a id='3'></a>

```
FASTQ bruts
    │
    ├─ [ÉTAPE 1] FastQC       → Rapport qualité
    ├─ [ÉTAPE 2] Trimmomatic  → Reads filtrés
    ├─ [ÉTAPE 3] MEGAHIT      → Contigs assemblés
    ├─ [ÉTAPE 4] Kraken2      → Profils taxonomiques
    ├─ [ÉTAPE 5] HUMAnN3      → Voies KEGG / MetaCyc
    └─────────────────────────────────────────────────
                              ↓
                     X_microbiome (n × p)
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PIPELINE BIOINFORMATIQUE — Wrappers Python
# En production : ces fonctions appellent les vrais outils via subprocess
# ══════════════════════════════════════════════════════════════════════

import subprocess, os

def run_fastqc(fastq_path: str, output_dir: str = 'qc_reports') -> dict:
    """
    ÉTAPE 1 — Contrôle qualité des reads bruts.

    Commande réelle :
        fastqc {fastq_path} -o {output_dir} --threads 8

    Retourne un rapport qualité simulé.
    """
    print(f"  [FastQC] Analyse qualité de : {fastq_path}")
    # En production : subprocess.run(['fastqc', fastq_path, '-o', output_dir])
    report = {
        'file'              : fastq_path,
        'total_reads'       : np.random.randint(800_000, 5_000_000),
        'reads_flagged_pct' : round(np.random.uniform(0.5, 8.0), 2),
        'mean_phred_score'  : round(np.random.uniform(28, 38), 1),
        'adapter_content'   : 'Warn' if np.random.rand() > 0.7 else 'Pass',
        'gc_content_pct'    : round(np.random.uniform(48, 56), 1),
        'pass'              : True,
    }
    return report


def run_trimmomatic(fastq_R1: str, fastq_R2: str,
                    min_len: int = 50, min_quality: int = 20) -> dict:
    """
    ÉTAPE 2 — Nettoyage des reads et suppression des adaptateurs.

    Commande réelle :
        trimmomatic PE {R1} {R2} {R1_out} {R1_unp} {R2_out} {R2_unp}
        ILLUMINACLIP:TruSeq3-PE.fa:2:30:10
        LEADING:{min_quality} TRAILING:{min_quality}
        SLIDINGWINDOW:4:{min_quality} MINLEN:{min_len}
    """
    print(f"  [Trimmomatic] Nettoyage : {fastq_R1} + {fastq_R2}")
    total = np.random.randint(800_000, 3_000_000)
    kept  = int(total * np.random.uniform(0.85, 0.98))
    return {
        'input_reads'   : total,
        'kept_reads'    : kept,
        'kept_pct'      : round(kept / total * 100, 2),
        'dropped_reads' : total - kept,
        'min_len'       : min_len,
        'min_quality'   : min_quality,
    }


def run_megahit(reads_dir: str, output_dir: str = 'megahit_out') -> dict:
    """
    ÉTAPE 3 — Assemblage métagénomique de novo.

    Commande réelle :
        megahit -1 R1.fastq -2 R2.fastq -o {output_dir}
        --min-contig-len 500 --k-min 21 --k-max 141 --k-step 12
        -t 16 --memory 0.8
    """
    print(f"  [MEGAHIT] Assemblage des contigs...")
    n_contigs = np.random.randint(50_000, 300_000)
    return {
        'n_contigs'       : n_contigs,
        'n50_bp'          : np.random.randint(500, 5_000),
        'total_length_bp' : n_contigs * np.random.randint(400, 2_000),
        'max_contig_bp'   : np.random.randint(20_000, 200_000),
        'output_fasta'    : f'{output_dir}/final.contigs.fa',
    }


def run_kraken2(contigs_fasta: str, db_path: str = '/data/kraken2_db',
                bacteria_names=None, n_species=200) -> pd.DataFrame:
    """
    ÉTAPE 4 — Classification taxonomique.

    Commande réelle :
        kraken2 --db {db_path} --paired --gzip-compressed
        --threads 16 --report report.txt
        --output output.txt {R1} {R2}

    Retourne : profil d'abondances relatives (Series)
    """
    print(f"  [Kraken2] Classification taxonomique : {contigs_fasta}")
    if bacteria_names is None:
        bacteria_names = [f'Bacterium_sp_{i}' for i in range(n_species)]
    alpha = np.random.exponential(0.5, len(bacteria_names)) + 0.01
    abundances = np.random.dirichlet(alpha)
    return pd.Series(abundances, index=bacteria_names, name='relative_abundance')


def run_humann3(classified_reads: str, db_path: str = '/data/uniref90',
                n_pathways: int = 300) -> pd.DataFrame:
    """
    ÉTAPE 5 — Annotation fonctionnelle (voies KEGG, MetaCyc, GO).

    Commande réelle :
        humann3 --input {classified_reads}
        --output {output_dir}
        --protein-database {db_path}
        --threads 16
        --metaphlan-options "--bowtie2db {bowtie2db}"

    Retourne : DataFrame des voies métaboliques (RPK normalisé)
    """
    print(f"  [HUMAnN3] Annotation fonctionnelle...")
    pathway_names = (
        [f'KEGG_K{i:05d}' for i in range(n_pathways // 3)] +
        [f'MetaCyc_PWY_{i}' for i in range(n_pathways // 3)] +
        [f'GO_{i}'          for i in range(n_pathways - 2*(n_pathways//3))]
    )
    # RPK = Reads Per Kilobase
    rpk = np.random.lognormal(mean=2.0, sigma=1.5, size=n_pathways)
    return pd.Series(rpk.clip(0), index=pathway_names, name='RPK')


# ── Test du pipeline sur un patient exemple ───────────────────────────
print("🔬 Test du pipeline bioinformatique (Patient_0001)\n")
print("=" * 55)

qc_report  = run_fastqc('patient_0001_R1.fastq.gz')
trim_stats = run_trimmomatic('patient_0001_R1.fastq.gz', 'patient_0001_R2.fastq.gz')
assembly   = run_megahit('trimmed_reads/')
taxonomy   = run_kraken2(assembly['output_fasta'], bacteria_names=BACTERIA_NAMES)
functions  = run_humann3('classified.fastq')

print("\n" + "=" * 55)
print("\n📊 Rapport pipeline Patient_0001 :")
print(f"  QC    → Phred moyen     : {qc_report['mean_phred_score']}")
print(f"  Trim  → Reads conservés : {trim_stats['kept_pct']}%")
print(f"  Assem → Contigs         : {assembly['n_contigs']:,}  (N50={assembly['n50_bp']} bp)")
print(f"  Taxo  → Espèces top 3   : {', '.join(taxonomy.nlargest(3).index.tolist())}")
print(f"  Funct → Voies détectées : {(functions > 0).sum()}")

In [ ]:
# ── Normalisation des abondances taxonomiques ─────────────────────────

def normalize_abundance(df_taxa: pd.DataFrame, method: str = 'relative') -> pd.DataFrame:
    """
    Normalise la matrice d'abondances.

    methods:
        'relative'  → abondances relatives (sum=1 par patient)
        'clr'       → Centered Log-Ratio (Aitchison, recommandé pour microbiome)
        'rarefy'    → Raréfaction à la profondeur minimale
    """
    if method == 'relative':
        row_sums = df_taxa.sum(axis=1)
        return df_taxa.div(row_sums, axis=0)

    elif method == 'clr':
        # Centred Log-Ratio — gère les zéros par pseudo-count
        df_pseudo = df_taxa + 0.5  # pseudo-count
        log_mat   = np.log(df_pseudo)
        geom_mean = log_mat.mean(axis=1)
        return log_mat.sub(geom_mean, axis=0)

    elif method == 'rarefy':
        # Raréfaction à la profondeur minimale
        min_depth = df_taxa.sum(axis=1).min()
        print(f"  Raréfaction à {int(min_depth)} reads")
        def rarefy_row(row):
            total   = int(row.sum())
            target  = int(min_depth)
            indices = np.repeat(np.arange(len(row)), row.astype(int).values)
            sampled = np.random.choice(indices, size=target, replace=False)
            counts  = np.bincount(sampled, minlength=len(row))
            return pd.Series(counts, index=row.index)
        return df_taxa.apply(rarefy_row, axis=1)

    else:
        raise ValueError(f"Méthode inconnue : {method}")


# Appliquer les normalisations
X_taxa_rel = normalize_abundance(X_taxa, method='relative')
X_taxa_clr = normalize_abundance(X_taxa, method='clr')

print("✅ Normalisation :")
print(f"   Relative → somme lignes : {X_taxa_rel.sum(axis=1).describe()[['min','max']].to_dict()}")
print(f"   CLR      → moyenne      : {X_taxa_clr.values.mean():.4f} (≈0 attendu)")
print(f"   Shape final : {X_taxa_rel.shape}")

---
## 4. Construction de la Matrice Globale <a id='4'></a>

$$X_{total} = [X_{microbiome} \mid X_{diet} \mid X_{symptoms}]$$

Dimension finale : $n \times p$ avec $n=$ patients et $p=$ features totales

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CONSTRUCTION DE LA MATRICE GLOBALE
# ══════════════════════════════════════════════════════════════════════

def build_global_matrix(
    X_microbiome: pd.DataFrame,
    X_diet: pd.DataFrame,
    X_clinical: pd.DataFrame,
    X_functional: pd.DataFrame = None,
    normalize: bool = True
) -> pd.DataFrame:
    """
    Construit la matrice globale par concaténation.

    X_total = [X_microbiome | X_functional | X_diet | X_clinical]
    """
    # Aligner les index
    common_idx = X_microbiome.index

    parts = {'microbiome': X_microbiome.loc[common_idx]}

    if X_functional is not None:
        parts['functional'] = X_functional.loc[common_idx]

    parts['diet']     = X_diet.loc[common_idx]
    parts['clinical'] = X_clinical.loc[common_idx]

    # Concaténation horizontale
    X_total = pd.concat(parts.values(), axis=1, keys=parts.keys())
    X_total.columns = ['_'.join(col) for col in X_total.columns]

    # Normalisation globale
    if normalize:
        scaler  = StandardScaler()
        X_array = scaler.fit_transform(X_total.fillna(0))
        X_total = pd.DataFrame(X_array, index=X_total.index, columns=X_total.columns)

    return X_total


# Sélectionner uniquement les colonnes numériques cliniques
X_symp_num = df_clin.select_dtypes(include=[np.number]).drop(
    columns=['diabete_risque', 'syndrome_metabolique', 'inflammation_elevee'],
    errors='ignore'
)

X_total = build_global_matrix(
    X_microbiome = X_taxa_clr,
    X_diet       = X_diet,
    X_clinical   = X_symp_num,
    X_functional = X_func,
    normalize    = True
)

n, p = X_total.shape
print("✅ Matrice globale construite :")
print(f"   n (patients)  = {n}")
print(f"   p (features)  = {p}")
print(f"   Dont microbiome  : {X_taxa_clr.shape[1]}")
print(f"   Dont fonctionnel : {X_func.shape[1]}")
print(f"   Dont alimentaire : {X_diet.shape[1]}")
print(f"   Dont clinique    : {X_symp_num.shape[1]}")
print(f"\n   Taille mémoire : {X_total.memory_usage(deep=True).sum() / 1e6:.2f} MB")

---
## 5. Réduction de Dimension <a id='5'></a>

$$X_{réduit} \leftarrow X_{microbiome} \quad (n \times k, \; k \ll p)$$

In [ ]:

# ══════════════════════════════════════════════════════════════════════
# 5A — PCA
# ══════════════════════════════════════════════════════════════════════

# PCA sur les données taxonomiques (CLR-transformées)
max_components = min(50, X_taxa_clr.shape[0] - 1, X_taxa_clr.shape[1])
if max_components < 2:
    raise ValueError(f"PCA impossible: max_components={max_components}")

pca = PCA(n_components=max_components, random_state=SEED)
X_taxa_pca = pca.fit_transform(X_taxa_clr.fillna(0))

cumvar = np.cumsum(pca.explained_variance_ratio_)
n_components_90 = np.argmax(cumvar >= 0.90) + 1

n_var_plot = min(20, max_components)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Analyse en Composantes Principales (PCA) — Microbiome',
             fontsize=14, fontweight='bold', color=BIOMEX_COLORS['navy'])

# Variance expliquée
axes[0].bar(range(1, n_var_plot + 1), pca.explained_variance_ratio_[:n_var_plot] * 100,
            color=BIOMEX_COLORS['blue'], alpha=0.8)
axes[0].set_xlabel('Composante Principale')
axes[0].set_ylabel('Variance expliquée (%)')
axes[0].set_title(f'Variance par composante ({n_var_plot} premières)')

# Variance cumulée
axes[1].plot(range(1, max_components + 1), cumvar * 100,
             color=BIOMEX_COLORS['green'], linewidth=2.5, marker='o', markersize=3)
axes[1].axhline(90, color='red', ls='--', label=f'90% → {n_components_90} CP')
axes[1].fill_between(range(1, max_components + 1), cumvar * 100, alpha=0.15,
                     color=BIOMEX_COLORS['green'])
axes[1].set_xlabel('Nombre de composantes')
axes[1].set_ylabel('Variance cumulée (%)')
axes[1].set_title('Variance cumulée — Choix du nombre de composantes')
axes[1].legend()

plt.tight_layout()
plt.savefig('02_pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ PCA : {n_components_90} composantes capturent 90% de la variance")
print(f"   Composantes testées : {max_components}")


In [ ]:
# ── Visualisation PCA — PC1 vs PC2 coloré par risque diabète ─────────
fig, ax = plt.subplots(figsize=(10, 7))

for label, color, name in [(0, BIOMEX_COLORS['green'], 'Faible risque'),
                            (1, BIOMEX_COLORS['orange'], 'Risque élevé')]:
    mask = Y['diabete_risque'] == label
    ax.scatter(X_taxa_pca[mask, 0], X_taxa_pca[mask, 1],
               c=color, alpha=0.6, s=40, label=name, edgecolors='white', lw=0.3)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
ax.set_title('PCA Microbiome — Séparation Risque Diabète', fontsize=14,
             fontweight='bold', color=BIOMEX_COLORS['navy'])
ax.legend(markerscale=1.5)

plt.savefig('03_pca_risque.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 5B — AUTOENCODER (Deep Learning)
# ══════════════════════════════════════════════════════════════════════

class BiomeXAutoencoder(nn.Module):
    """
    Autoencoder PyTorch pour réduction non-linéaire du microbiome.

    Architecture : Encoder → Espace latent z → Decoder
    X (haute dim) → z (dim réduite) → X̂ (reconstruction)

    Perte : MSE(X, X̂)  — minimise l'erreur de reconstruction
    """
    def __init__(self, input_dim: int, latent_dim: int = 32, hidden_dims=(256, 128)):
        super().__init__()

        # Encoder
        enc_layers = []
        prev_dim   = input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(prev_dim, h), nn.BatchNorm1d(h),
                           nn.ReLU(), nn.Dropout(0.2)]
            prev_dim = h
        enc_layers.append(nn.Linear(prev_dim, latent_dim))
        self.encoder = nn.Sequential(*enc_layers)

        # Decoder (miroir)
        dec_layers = []
        prev_dim   = latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [nn.Linear(prev_dim, h), nn.BatchNorm1d(h),
                           nn.ReLU(), nn.Dropout(0.2)]
            prev_dim = h
        dec_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*dec_layers)

    def forward(self, x):
        z    = self.encoder(x)          # z = Encoder(X)
        x_hat = self.decoder(z)         # X̂ = Decoder(z)
        return x_hat, z

    def encode(self, x):
        return self.encoder(x)


def train_autoencoder(X_data: np.ndarray, latent_dim=32,
                      n_epochs=50, batch_size=64, lr=1e-3,
                      device='cpu') -> tuple:
    """
    Entraîne l'autoencoder et retourne le modèle + représentations latentes.
    """
    X_tensor = torch.FloatTensor(X_data).to(device)
    dataset  = TensorDataset(X_tensor)
    loader   = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model     = BiomeXAutoencoder(X_data.shape[1], latent_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    losses = []
    model.train()

    for epoch in range(n_epochs):
        epoch_loss = 0
        for (batch,) in loader:
            optimizer.zero_grad()
            x_hat, _ = model(batch)
            loss      = criterion(x_hat, batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch)

        epoch_loss /= len(X_data)
        losses.append(epoch_loss)
        scheduler.step()

        if (epoch + 1) % 10 == 0:
            print(f"   Epoch {epoch+1:3d}/{n_epochs} | MSE Loss : {epoch_loss:.6f}")

    # Extraction des représentations latentes
    model.eval()
    with torch.no_grad():
        _, Z = model(X_tensor)
    Z_np = Z.cpu().numpy()

    return model, Z_np, losses


# Entraînement sur données taxonomiques
print("🧠 Entraînement de l'Autoencoder BiomeX...")
LATENT_DIM = 32

X_ae_input = X_taxa_clr.fillna(0).values.astype(np.float32)
scaler_ae  = StandardScaler()
X_ae_input = scaler_ae.fit_transform(X_ae_input)

ae_model, Z_latent, ae_losses = train_autoencoder(
    X_ae_input, latent_dim=LATENT_DIM, n_epochs=50, batch_size=32
)

# Visualisation de l'entraînement
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ae_losses, color=BIOMEX_COLORS['purple'], linewidth=2)
ax.fill_between(range(len(ae_losses)), ae_losses, alpha=0.2,
                color=BIOMEX_COLORS['purple'])
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Autoencoder — Courbe d\'apprentissage', fontsize=14,
             fontweight='bold', color=BIOMEX_COLORS['navy'])
plt.savefig('04_autoencoder_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Espace latent z : {Z_latent.shape} (patients × {LATENT_DIM} dimensions latentes)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 5C — Sélection par importance (Random Forest Feature Selection)
# ══════════════════════════════════════════════════════════════════════

rf_selector = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf_selector.fit(X_taxa_clr.fillna(0), Y['diabete_risque'])

importances = pd.Series(
    rf_selector.feature_importances_,
    index=X_taxa_clr.columns
).sort_values(ascending=False)

top_k = 30
selected_features = importances.head(top_k).index.tolist()
X_taxa_selected   = X_taxa_clr[selected_features]

# Visualisation top 20 features
fig, ax = plt.subplots(figsize=(12, 8))
colors  = [BIOMEX_COLORS['orange'] if f in PATHOBIONTES
           else BIOMEX_COLORS['green'] if f in BUTYRATE_PRODUCERS
           else BIOMEX_COLORS['blue']
           for f in importances.head(20).index]

importances.head(20).plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_xlabel('Importance MDI (Mean Decrease Impurity)')
ax.set_title('Top 20 Bactéries — Prédiction Risque Diabète', fontsize=14,
             fontweight='bold', color=BIOMEX_COLORS['navy'])
ax.invert_yaxis()

patches = [
    mpatches.Patch(color=BIOMEX_COLORS['orange'], label='Pathobionte'),
    mpatches.Patch(color=BIOMEX_COLORS['green'],  label='Producteur SCFA'),
    mpatches.Patch(color=BIOMEX_COLORS['blue'],   label='Autre'),
]
ax.legend(handles=patches, loc='lower right')

plt.tight_layout()
plt.savefig('05_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Sélection : {top_k} features les plus prédictives retenues")

---
## 6. Calcul des Scores Biologiques <a id='6'></a>

| Score | Formule |
|---|---|
| Shannon H | $H = -\sum_i p_i \log(p_i)$ |
| Inflammation | $S_{inflam} = \sum_i w_i \cdot Abondance_i$ |
| SCFA | $S_{SCFA} = \sum Abondance_{gènes\_butyrate}$ |
| Dysbiose | $D = \frac{\sum pathobiontes}{\sum bactéries\_bénéfiques}$ |
| Score Global | $S_{met} = \alpha H + \beta S_{SCFA} - \gamma D - \delta S_{inflam}$ |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CLASSE : BiomeX Score Engine
# ══════════════════════════════════════════════════════════════════════

class BiomeXScoreEngine:
    """
    Calcule les 5 scores biologiques BiomeX à partir des abondances.

    Scores produits :
        - shannon_H       : Diversité microbienne (Indice de Shannon)
        - score_inflam    : Inflammation microbienne pondérée
        - score_scfa      : Production estimée d'AGCC
        - score_dysbiose  : Ratio pathobiontes / bactéries bénéfiques
        - score_global    : Score métabolique global normalisé 0–100
    """

    # Poids inflammation (basés sur la littérature — valeurs positives = pro-inflam)
    INFLAMMATION_WEIGHTS = {
        'Escherichia_coli'       :  2.0,
        'Klebsiella_pneumoniae'  :  1.8,
        'Helicobacter_pylori'    :  2.5,
        'Enterobacter_cloacae'   :  1.5,
        'Ruminococcus_gnavus'    :  1.2,
        'Faecalibacterium_prausnitzii' : -3.0,  # Anti-inflammatoire !
        'Lactobacillus_rhamnosus'      : -2.0,
        'Bifidobacterium_longum'       : -1.8,
        'Akkermansia_muciniphila'      : -2.5,
    }

    BENEFICIAL_BACTERIA = [
        'Faecalibacterium_prausnitzii', 'Lactobacillus_rhamnosus',
        'Bifidobacterium_longum', 'Akkermansia_muciniphila',
        'Roseburia_intestinalis', 'Lactobacillus_acidophilus',
        'Bifidobacterium_adolescentis'
    ]

    def __init__(self, alpha=0.3, beta=0.3, gamma=0.2, delta=0.2):
        """Coefficients de pondération du score global."""
        self.alpha = alpha  # Poids diversité
        self.beta  = beta   # Poids SCFA
        self.gamma = gamma  # Poids dysbiose (négatif)
        self.delta = delta  # Poids inflammation (négatif)

    # ── Score 1 : Diversité (Shannon H) ──────────────────────────────
    def shannon_diversity(self, abundances: pd.Series) -> float:
        """
        H = -Σᵢ pᵢ · log(pᵢ)

        pᵢ = proportion relative de la bactérie i
        Plus H est élevé → microbiome plus diversifié et résilient
        Valeur typique : 1.5 (faible) à 4.5 (haute diversité)
        """
        p = abundances[abundances > 0]
        p = p / p.sum()
        return float(-np.sum(p * np.log(p)))

    # ── Score 2 : Inflammation ────────────────────────────────────────
    def inflammation_score(self, abundances: pd.Series) -> float:
        """
        S_inflam = Σᵢ wᵢ × Abondance_i

        wᵢ = poids de la bactérie i selon la littérature
        Positif = pro-inflammatoire · Négatif = anti-inflammatoire
        """
        score = 0.0
        for bacteria, weight in self.INFLAMMATION_WEIGHTS.items():
            if bacteria in abundances.index:
                score += weight * abundances[bacteria]
        # Normalisation Min-Max → [0, 1]
        return float(np.clip(score, -5, 5))  # clip pour stabilité

    # ── Score 3 : SCFA (production d'acides gras courts) ─────────────
    def scfa_score(self, abundances: pd.Series,
                   functional_data: pd.Series = None) -> float:
        """
        S_SCFA = Σ Abondance(gènes_butyrate) + Σ Abondance(gènes_propionate)

        En production : utiliser les données HUMAnN3 (gènes K00929, K01902...)
        Ici : proxy via abondance des producteurs connus
        """
        if functional_data is not None:
            # Gènes butyrate kinase (K00929) + phosphotransbutyrylase (K00634)
            butyrate_genes = [c for c in functional_data.index
                             if 'K00929' in c or 'K00634' in c or 'K01902' in c]
            if butyrate_genes:
                return float(functional_data[butyrate_genes].sum())

        # Proxy taxonomique
        scfa_bacteria = [b for b in BUTYRATE_PRODUCERS if b in abundances.index]
        return float(abundances[scfa_bacteria].sum()) if scfa_bacteria else 0.0

    # ── Score 4 : Dysbiose ────────────────────────────────────────────
    def dysbiosis_score(self, abundances: pd.Series) -> float:
        """
        D = Σ(pathobiontes) / Σ(bactéries bénéfiques)

        D < 1.0  = équilibre microbien (optimal)
        D > 1.5  = dysbiose cliniquement significative
        D > 2.0  = intervention recommandée
        """
        patho_present = [p for p in PATHOBIONTES if p in abundances.index]
        bene_present  = [b for b in self.BENEFICIAL_BACTERIA if b in abundances.index]

        patho_sum = abundances[patho_present].sum() if patho_present else 1e-10
        bene_sum  = abundances[bene_present].sum()  if bene_present  else 1e-10

        return float(patho_sum / (bene_sum + 1e-10))

    # ── Score 5 : Score Métabolique Global ────────────────────────────
    def global_score(self, H: float, scfa: float, dysbiose: float,
                     inflam: float) -> float:
        """
        S_met = α·H + β·S_SCFA - γ·D - δ·S_inflam

        Coefficients α, β, γ, δ optimisés par validation clinique.
        Normalisé 0–100 pour lisibilité patient.
        """
        raw = (self.alpha * H +
               self.beta  * scfa -
               self.gamma * dysbiose -
               self.delta * inflam)
        # Normalisation empirique vers [0, 100]
        normalized = np.clip((raw + 3) / 6 * 100, 0, 100)
        return float(normalized)

    # ── Calcul sur toute la cohorte ───────────────────────────────────
    def compute_all_scores(self, X_taxa_rel: pd.DataFrame,
                           X_func: pd.DataFrame = None) -> pd.DataFrame:
        """Calcule les 5 scores pour chaque patient de la cohorte."""
        from tqdm import tqdm
        scores = []

        for patient_id in tqdm(X_taxa_rel.index, desc='Calcul des scores'):
            abundances = X_taxa_rel.loc[patient_id]
            func_data  = X_func.loc[patient_id] if X_func is not None else None

            H        = self.shannon_diversity(abundances)
            inflam   = self.inflammation_score(abundances)
            scfa     = self.scfa_score(abundances, func_data)
            dysbiose = self.dysbiosis_score(abundances)
            glob     = self.global_score(H, scfa, dysbiose, inflam)

            scores.append({
                'patient_id'      : patient_id,
                'shannon_H'       : round(H, 4),
                'score_inflam'    : round(inflam, 4),
                'score_scfa'      : round(scfa, 6),
                'score_dysbiose'  : round(dysbiose, 4),
                'score_global'    : round(glob, 2),
            })

        df_scores = pd.DataFrame(scores).set_index('patient_id')

        # Catégorisation du score global
        df_scores['niveau_sante'] = pd.cut(
            df_scores['score_global'],
            bins=[0, 33, 66, 100],
            labels=['Préoccupant', 'Modéré', 'Optimal']
        )
        return df_scores


# ── Calcul des scores ─────────────────────────────────────────────────
print("⚙️  Calcul des scores biologiques BiomeX...")
engine   = BiomeXScoreEngine(alpha=0.3, beta=0.3, gamma=0.2, delta=0.2)
df_scores = engine.compute_all_scores(X_taxa_rel, X_func)

print("\n✅ Scores calculés :")
print(df_scores.describe().round(3).to_string())

In [ ]:
# ── Visualisation des scores ──────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('BiomeX — Distribution des 5 Scores Biologiques (Cohorte Pilote)',
             fontsize=16, fontweight='bold', color=BIOMEX_COLORS['navy'])

scores_config = [
    ('shannon_H',      'Diversité Shannon (H)',     BIOMEX_COLORS['green'],  '0 (faible) → 4+ (optimal)', 0, 0),
    ('score_inflam',   'Score Inflammation',        BIOMEX_COLORS['orange'], 'Négatif = anti-inflam',      0, 1),
    ('score_scfa',     'Score SCFA (Butyrate)',     BIOMEX_COLORS['teal'],   'Plus élevé = meilleure santé colique', 0, 2),
    ('score_dysbiose', 'Score Dysbiose',            BIOMEX_COLORS['purple'], '< 1.0 = optimal · > 1.5 = dysbiose',  1, 0),
    ('score_global',   'Score Métabolique Global',  BIOMEX_COLORS['blue'],   '0 (faible) → 100 (optimal)',  1, 1),
]

for col, title, color, subtitle, row, cidx in scores_config:
    ax = axes[row][cidx]
    ax.hist(df_scores[col], bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(df_scores[col].mean(), color='red', ls='--',
               label=f'Moyenne : {df_scores[col].mean():.2f}')
    ax.set_title(f'{title}\n{subtitle}', fontsize=11)
    ax.legend(fontsize=9)

# Pie chart répartition niveaux santé
ax_pie = axes[1][2]
counts = df_scores['niveau_sante'].value_counts()
ax_pie.pie(counts.values, labels=counts.index,
           colors=[BIOMEX_COLORS['orange'], BIOMEX_COLORS['blue'], BIOMEX_COLORS['green']],
           autopct='%1.1f%%', startangle=90,
           wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax_pie.set_title('Répartition Niveaux de Santé', fontsize=11)

plt.tight_layout()
plt.savefig('06_scores_biologiques.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Corrélation scores vs clinique ───────────────────────────────────
df_combined = pd.concat([
    df_scores[['shannon_H', 'score_inflam', 'score_dysbiose', 'score_global']],
    df_clin[['glycemie_mmol', 'IMC', 'CRP_mg_L', 'HbA1c_pct']]
], axis=1)

corr = df_combined.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, ax=ax, mask=mask,
            cbar_kws={'label': 'Coefficient de corrélation'},
            annot_kws={'size': 10})
ax.set_title('Corrélations : Scores BiomeX × Biomarqueurs Cliniques',
             fontsize=13, fontweight='bold', color=BIOMEX_COLORS['navy'])
plt.tight_layout()
plt.savefig('07_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Modélisation Prédictive Supervisée <a id='7'></a>

| Modèle | Formule | Cible |
|---|---|---|
| Régression Logistique | $P(Y=1\|X) = \frac{1}{1+e^{-\beta X}}$ | Risque maladies chroniques |
| Random Forest | $\hat{Y} = \text{mode}(\text{arbre}_1...\text{arbre}_n)$ | Inflammation, glycémie |
| XGBoost | $\hat{Y} = \sum_k f_k(X)$ | Score métabolique global |

In [ ]:

# ══════════════════════════════════════════════════════════════════════
# PRÉPARATION DES FEATURES ENRICHIES
# ══════════════════════════════════════════════════════════════════════

# Combiner : PCA (n_components_90) + Scores biologiques (5) + Clinique (num)
df_pca = pd.DataFrame(
    X_taxa_pca[:, :n_components_90],
    index=X_taxa_rel.index,
    columns=[f'PC{i+1}' for i in range(n_components_90)]
)

X_features = pd.concat([
    df_pca,
    df_scores[['shannon_H', 'score_inflam', 'score_scfa',
               'score_dysbiose', 'score_global']],
    X_symp_num,
], axis=1).fillna(0)

print(f"✅ Feature matrix : {X_features.shape}")
print(f"   Dont PCA components  : {n_components_90}")
print(f"   Dont scores BiomeX   : 5")
print(f"   Dont variables cliniq.: {X_symp_num.shape[1]}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ENTRAÎNEMENT DES 3 MODÈLES
# ══════════════════════════════════════════════════════════════════════

TARGET = 'diabete_risque'

X = X_features.values
y = Y[TARGET].values

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Train : {X_train.shape[0]} patients | Test : {X_test.shape[0]} patients")
print(f"Prévalence train : {y_train.mean():.2%} | test : {y_test.mean():.2%}\n")

# ── Modèle 1 : Régression Logistique ─────────────────────────────────
print("📊 Entraînement Régression Logistique...")
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=1.0, max_iter=1000,
                                  class_weight='balanced',
                                  random_state=SEED))
])
lr_model.fit(X_train, y_train)

# ── Modèle 2 : Random Forest ──────────────────────────────────────────
print("🌲 Entraînement Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=500, max_depth=10, min_samples_leaf=3,
    class_weight='balanced', random_state=SEED, n_jobs=-1
)
rf_model.fit(X_train, y_train)

# ── Modèle 3 : XGBoost ───────────────────────────────────────────────
print("⚡ Entraînement XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    use_label_encoder=False, eval_metric='logloss',
    random_state=SEED, n_jobs=-1
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print("\n✅ 3 modèles entraînés")

In [ ]:
# ── Évaluation comparée des modèles ──────────────────────────────────

def evaluate_model(model, X_test, y_test, name):
    """Évalue un modèle et retourne ses métriques principales."""
    y_pred      = model.predict(X_test)
    y_proba     = model.predict_proba(X_test)[:, 1]
    auc         = roc_auc_score(y_test, y_proba)
    f1          = f1_score(y_test, y_pred, average='weighted')
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    return {'name': name, 'AUC': auc, 'F1': f1,
            'fpr': fpr, 'tpr': tpr, 'proba': y_proba}

results = [
    evaluate_model(lr_model,  X_test, y_test, 'Régression Logistique'),
    evaluate_model(rf_model,  X_test, y_test, 'Random Forest'),
    evaluate_model(xgb_model, X_test, y_test, 'XGBoost'),
]

# Tableau de comparaison
df_results = pd.DataFrame([{'Modèle': r['name'], 'AUC-ROC': r['AUC'], 'F1-Score': r['F1']}
                            for r in results]).sort_values('AUC-ROC', ascending=False)
print("\n📊 Comparaison des modèles — Prédiction Risque Diabète\n")
print(df_results.to_string(index=False))

# Courbes ROC
fig, ax = plt.subplots(figsize=(9, 7))
colors  = [BIOMEX_COLORS['blue'], BIOMEX_COLORS['green'], BIOMEX_COLORS['orange']]

for r, color in zip(results, colors):
    ax.plot(r['fpr'], r['tpr'], color=color, linewidth=2.5,
            label=f"{r['name']} (AUC = {r['AUC']:.3f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5, label='Aléatoire (AUC = 0.5)')
ax.fill_between(results[-1]['fpr'], results[-1]['tpr'], alpha=0.08,
                color=BIOMEX_COLORS['orange'])

ax.set_xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
ax.set_ylabel('Taux de Vrais Positifs (TPR / Sensibilité)', fontsize=12)
ax.set_title('Courbes ROC — Prédiction Risque Diabète Type 2', fontsize=14,
             fontweight='bold', color=BIOMEX_COLORS['navy'])
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('08_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cross-validation K-Fold ───────────────────────────────────────────

print("🔁 Cross-Validation K-Fold (k=5)...")
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_results = {}
for name, model in [('Régression Logistique', lr_model),
                    ('Random Forest',         rf_model),
                    ('XGBoost',               xgb_model)]:
    scores = cross_val_score(model, X, y, cv=kfold,
                             scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f"  {name:28s} : {scores.mean():.3f} ± {scores.std():.3f}")

# Boxplot CV
fig, ax = plt.subplots(figsize=(10, 5))
data    = [cv_results[k] for k in cv_results]
labels  = list(cv_results.keys())
bp      = ax.boxplot(data, labels=labels, patch_artist=True,
                     boxprops=dict(facecolor=BIOMEX_COLORS['light'], color=BIOMEX_COLORS['navy']),
                     medianprops=dict(color=BIOMEX_COLORS['green'], linewidth=2.5),
                     whiskerprops=dict(color=BIOMEX_COLORS['navy']),
                     capprops=dict(color=BIOMEX_COLORS['navy']))

ax.axhline(0.85, color='red', ls='--', alpha=0.7, label='Seuil cible BiomeX (0.85)')
ax.set_ylabel('AUC-ROC')
ax.set_title('Cross-Validation K-Fold (k=5) — Stabilité des Modèles', fontsize=13,
             fontweight='bold', color=BIOMEX_COLORS['navy'])
ax.set_ylim([0.5, 1.0])
ax.legend()

plt.tight_layout()
plt.savefig('09_cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Prédiction pour un Nouvel Individu <a id='8'></a>

$$\hat{Y} = f(X_{nouveau}) \rightarrow \{\text{risque inflam.}, \text{glycémique}, D, S_{SCFA}, H, \text{recommandations}\}$$

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PIPELINE DE PRÉDICTION — NOUVEL INDIVIDU
# ══════════════════════════════════════════════════════════════════════

class BiomeXPredictor:
    """
    Système de prédiction end-to-end pour un nouvel individu.

    Ŷ = f(X_nouveau) → 7 outputs cliniques + recommandations
    """

    RISK_THRESHOLDS = {
        'faible'  : (0.0, 0.25),
        'modéré'  : (0.25, 0.60),
        'élevé'   : (0.60, 1.0),
    }

    FOOD_RECOMMENDATIONS = {
        'low_scfa'    : [
            "🫘 Augmenter le niébé (haricots africains) — 2× par semaine",
            "🌾 Intégrer le pain de mil ou le couscous de sorgho",
            "🍃 Consommer des feuilles de baobab séchées",
            "🥜 Arachides non transformées avec leur peau (fibres)",
        ],
        'high_inflam' : [
            "🐟 Augmenter le poisson (tilapia, capitaine) — oméga-3 anti-inflammatoires",
            "🫚 Remplacer les huiles transformées par l'huile de palme rouge (modérément)",
            "🧄 Ail frais quotidiennement — allicine anti-inflammatoire",
            "🥤 Jus de tamarin frais — polyphénols antioxydants",
        ],
        'dysbiosis'   : [
            "🥛 Yaourt fermenté local (lait caillé sénégalais) — probiotiques naturels",
            "🌽 Son de mil ou de sorgho — prébiotique puissant",
            "🫙 Légumes fermentés traditionnels (carottes, chou)",
            "🚫 Réduire les aliments ultra-transformés et les sucres ajoutés",
        ],
        'low_diversity': [
            "🌈 Diversifier les légumes — minimum 5 variétés différentes par semaine",
            "🫐 Fruits locaux variés : mangue, papaye, goyave, tamarin",
            "🌿 Herbes fraîches dans les plats : persil, coriandre, basilic africain",
        ],
    }

    PROBIOTIC_RECOMMENDATIONS = {
        'low_lactobacillus' : [
            "Lactobacillus rhamnosus GG — 10⁹ UFC/jour pendant 30 jours",
            "Lactobacillus acidophilus LA-5 — disponible en pharmacie",
        ],
        'low_bifido'        : [
            "Bifidobacterium longum BB536 — 10⁸ UFC/jour",
            "Bifidobacterium adolescentis — souche spécifique Afrique de l'Ouest",
        ],
        'high_dysbiosis'    : [
            "Saccharomyces boulardii — protection contre Clostridium difficile",
            "Akkermansia muciniphila (pasteurisée) — barrière intestinale",
        ],
    }

    def __init__(self, rf_model, xgb_model, lr_model, pca, scaler_ae,
                 score_engine, bacteria_names, n_pca_components):
        self.rf_model        = rf_model
        self.xgb_model       = xgb_model
        self.lr_model        = lr_model
        self.pca             = pca
        self.scaler_ae       = scaler_ae
        self.score_engine    = score_engine
        self.bacteria_names  = bacteria_names
        self.n_pca           = n_pca_components

    def _classify_risk(self, proba: float) -> str:
        for level, (lo, hi) in self.RISK_THRESHOLDS.items():
            if lo <= proba < hi:
                return level
        return 'élevé'

    def predict(self, taxa_vector: pd.Series, clinical_data: dict,
                diet_data: dict = None) -> dict:
        """
        Prédit les scores et risques pour un nouvel individu.

        Args:
            taxa_vector   : abondances relatives (Series, indexée par bactérie)
            clinical_data : dict de biomarqueurs cliniques
            diet_data     : dict de données alimentaires (optionnel)

        Returns:
            dict complet avec scores, risques, recommandations
        """

        # 1. Calcul des scores biologiques
        H        = self.score_engine.shannon_diversity(taxa_vector)
        inflam   = self.score_engine.inflammation_score(taxa_vector)
        scfa     = self.score_engine.scfa_score(taxa_vector)
        dysbiose = self.score_engine.dysbiosis_score(taxa_vector)
        glob     = self.score_engine.global_score(H, scfa, dysbiose, inflam)

        # 2. Préparation des features pour les modèles ML
        taxa_clr   = np.log(taxa_vector.clip(lower=1e-10) + 0.5)
        taxa_clr  -= taxa_clr.mean()
        pca_vec    = self.pca.transform(taxa_clr.values.reshape(1, -1))[0, :self.n_pca]

        clin_vec  = np.array([
            clinical_data.get('age',           35),
            clinical_data.get('sexe',           0),
            clinical_data.get('IMC',           25),
            clinical_data.get('glycemie_mmol',  5.5),
            clinical_data.get('HbA1c_pct',      5.5),
            clinical_data.get('CRP_mg_L',       2.0),
            clinical_data.get('tension_sys',   120),
            clinical_data.get('tension_dia',    80),
            clinical_data.get('cholesterol',    4.8),
            clinical_data.get('triglycerides',  1.5),
        ])

        # Feature vector complet
        scores_vec = np.array([H, inflam, scfa, dysbiose, glob])
        padding    = np.zeros(max(0, X_features.shape[1] -
                                  self.n_pca - 5 - len(clin_vec)))
        x_new      = np.concatenate([pca_vec, scores_vec, clin_vec, padding])
        x_new      = x_new[:X_features.shape[1]].reshape(1, -1)

        # 3. Prédictions des modèles
        proba_rf  = self.rf_model.predict_proba(x_new)[0, 1]
        proba_xgb = self.xgb_model.predict_proba(x_new)[0, 1]
        proba_lr  = self.lr_model.predict_proba(x_new)[0, 1]

        # Ensemble (moyenne pondérée)
        proba_final = 0.4 * proba_xgb + 0.4 * proba_rf + 0.2 * proba_lr

        # 4. Génération des recommandations
        recs = self._generate_recommendations(H, inflam, scfa, dysbiose,
                                               taxa_vector, clinical_data)

        return {
            '🧬 Scores Biologiques': {
                'Diversité Shannon H'      : f"{H:.2f}  {'✅ Bon' if H > 3 else '⚠️ Faible' if H > 2 else '🔴 Critique'}",
                'Score Inflammation'       : f"{inflam:.2f}  {'✅ Faible' if inflam < 0 else '⚠️ Modéré' if inflam < 1 else '🔴 Élevé'}",
                'Score SCFA (Butyrate)'    : f"{scfa:.4f}  {'✅ Bon' if scfa > 0.05 else '⚠️ Faible'}",
                'Score Dysbiose'           : f"{dysbiose:.2f}  {'✅ Équilibré' if dysbiose < 1.0 else '⚠️ Modéré' if dysbiose < 1.5 else '🔴 Dysbiose'}",
                'Score Global Métabolique' : f"{glob:.1f}/100  {'🟢' if glob > 66 else '🟡' if glob > 33 else '🔴'}",
            },
            '⚕️ Prédictions Cliniques': {
                'Risque Diabète T2 (RF)'   : f"{proba_rf:.1%}  [{self._classify_risk(proba_rf)}]",
                'Risque Diabète T2 (XGB)'  : f"{proba_xgb:.1%}  [{self._classify_risk(proba_xgb)}]",
                'Risque Diabète T2 (LR)'   : f"{proba_lr:.1%}  [{self._classify_risk(proba_lr)}]",
                'Score Ensemble Final'     : f"{proba_final:.1%}  [{self._classify_risk(proba_final)}]",
            },
            '🥗 Recommandations Nutritionnelles': recs['alimentation'],
            '🦠 Recommandations Probiotiques'   : recs['probiotiques'],
            '🌙 Recommandations Mode de Vie'    : recs['mode_de_vie'],
            '⚠️ Alertes Cliniques'              : recs['alertes'],
        }

    def _generate_recommendations(self, H, inflam, scfa, dysbiose,
                                   taxa, clinical) -> dict:
        """Génère des recommandations ciblées basées sur les scores."""
        alim, probio, lifestyle, alertes = [], [], [], []

        # Alimentation
        if scfa < 0.05:
            alim += self.FOOD_RECOMMENDATIONS['low_scfa'][:2]
        if inflam > 0.5:
            alim += self.FOOD_RECOMMENDATIONS['high_inflam'][:2]
        if dysbiose > 1.0:
            alim += self.FOOD_RECOMMENDATIONS['dysbiosis'][:2]
        if H < 2.5:
            alim += self.FOOD_RECOMMENDATIONS['low_diversity'][:2]

        # Probiotiques
        lact_present = any('Lactobacillus' in b for b in taxa[taxa > 0.01].index)
        if not lact_present:
            probio += self.PROBIOTIC_RECOMMENDATIONS['low_lactobacillus']
        if dysbiose > 1.5:
            probio += self.PROBIOTIC_RECOMMENDATIONS['high_dysbiosis']

        # Mode de vie
        lifestyle = [
            f"😴 Sommeil : {'✅ Cible atteinte' if clinical.get('age',35) < 65 else '⚠️ Améliorer'} — 7–9h recommandées (axe intestin-cerveau)",
            "🚶 Activité : 30 min de marche rapide × 5 jours/semaine — améliore la diversité microbienne",
            "🧘 Stress chronique : techniques de respiration ou yoga — cortisol élevé nuit au microbiome",
            "🚫 Antibiotiques : éviter l'automédication — usage excessif détruit 30–50% de la flore",
        ]

        # Alertes
        if clinical.get('glycemie_mmol', 5) > 7.0:
            alertes.append("🔴 URGENT — Glycémie > 7 mmol/L : consulter un médecin immédiatement")
        if dysbiose > 2.0:
            alertes.append("🔴 Dysbiose sévère (D > 2.0) : consultation gastro-entérologue recommandée")
        if H < 1.5:
            alertes.append("⚠️ Diversité microbienne critique — risque élevé de maladies chroniques")

        return {'alimentation': alim or ["✅ Alimentation équilibrée — continuer ainsi"],
                'probiotiques': probio or ["✅ Profil microbien satisfaisant"],
                'mode_de_vie' : lifestyle,
                'alertes'     : alertes or ["✅ Aucune alerte clinique"]}


# ── Instanciation du prédicteur ───────────────────────────────────────
predictor = BiomeXPredictor(
    rf_model        = rf_model,
    xgb_model       = xgb_model,
    lr_model        = lr_model,
    pca             = pca,
    scaler_ae       = scaler_ae,
    score_engine    = engine,
    bacteria_names  = BACTERIA_NAMES,
    n_pca_components= n_components_90,
)
print("✅ BiomeXPredictor instancié et prêt")

In [ ]:

# ── Test sur un patient réel de la cohorte ────────────────────────────
preferred_patient_id = 'Patient_0042'
patient_id = preferred_patient_id if preferred_patient_id in X_taxa_rel.index else X_taxa_rel.index[0]

if patient_id != preferred_patient_id:
    print(f"ℹ️ {preferred_patient_id} absent de la cohorte active, utilisation de {patient_id}")

taxa_new       = X_taxa_rel.loc[patient_id]
clinical_new   = df_clin.loc[patient_id].to_dict()

print(f"
{'='*60}")
print(f"  RAPPORT BIOMEX — {patient_id}")
print(f"{'='*60}
")

report = predictor.predict(taxa_new, clinical_new)

for section, content in report.items():
    print(f"
{section}")
    print("─" * 50)
    if isinstance(content, dict):
        for k, v in content.items():
            print(f"  {k:<35} : {v}")
    elif isinstance(content, list):
        for item in content:
            print(f"  • {item}")

print(f"
{'='*60}")


---
## 9. Validation et Interprétabilité (SHAP) <a id='9'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SHAP VALUES — Interprétabilité Clinique
# ══════════════════════════════════════════════════════════════════════

print("🔍 Calcul des SHAP values (XGBoost)...")
feature_names = X_features.columns.tolist()

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test[:100])  # Sous-ensemble pour rapidité

# Global feature importance via SHAP
fig = plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values, X_test[:100],
    feature_names=feature_names,
    max_display=20,
    plot_type='bar',
    color=BIOMEX_COLORS['green'],
    show=False
)
plt.title('Importance Globale des Features — SHAP (XGBoost)', fontsize=14,
          fontweight='bold', color=BIOMEX_COLORS['navy'])
plt.tight_layout()
plt.savefig('10_shap_global.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Sauvegardé : 10_shap_global.png")

In [ ]:
# ── SHAP Beeswarm — Impact et Direction ──────────────────────────────
fig = plt.figure(figsize=(12, 9))
shap.summary_plot(
    shap_values, X_test[:100],
    feature_names=feature_names,
    max_display=15,
    show=False
)
plt.title('SHAP Beeswarm — Direction et Magnitude des Features', fontsize=14,
          fontweight='bold', color=BIOMEX_COLORS['navy'])
plt.tight_layout()
plt.savefig('11_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP Force Plot — Explication individuelle ────────────────────────
patient_idx = 5
print(f"\n🔎 Explication SHAP individuelle — Patient test #{patient_idx}")
print(f"   Prédiction : {xgb_model.predict_proba(X_test[patient_idx:patient_idx+1])[0,1]:.1%} de risque diabète")

# Top features qui augmentent ou réduisent le risque
shap_patient = shap_values[patient_idx]
top_pos = pd.Series(shap_patient, index=feature_names).nlargest(5)
top_neg = pd.Series(shap_patient, index=feature_names).nsmallest(5)

print("\n  🔴 Facteurs augmentant le risque :")
for feat, val in top_pos.items():
    print(f"     +{val:.4f}  ←  {feat}")

print("\n  🟢 Facteurs réduisant le risque :")
for feat, val in top_neg.items():
    print(f"     {val:.4f}  ←  {feat}")

In [ ]:
# ── Métriques complètes de validation ────────────────────────────────

print("\n" + "="*60)
print("  RAPPORT DE VALIDATION COMPLET — BiomeX v4.0")
print("="*60)

best_model = xgb_model  # XGBoost généralement le plus performant
y_pred     = best_model.predict(X_test)
y_proba    = best_model.predict_proba(X_test)[:, 1]

auc  = roc_auc_score(y_test, y_proba)
rmse = np.sqrt(mean_squared_error(y_test, y_proba))
brier = np.mean((y_proba - y_test) ** 2)

print(f"\n  Modèle champion   : XGBoost")
print(f"  AUC-ROC           : {auc:.4f}  {'✅' if auc > 0.85 else '⚠️'}  (cible BiomeX : > 0.85)")
print(f"  RMSE              : {rmse:.4f}")
print(f"  Brier Score       : {brier:.4f}  (calibration, < 0.25 = bon)")
print(f"  F1-Score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")

print(f"\n  Rapport de Classification :")
print(classification_report(y_test, y_pred,
                            target_names=['Faible risque', 'Risque élevé']))

# Matrice de confusion
cm  = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Prédit : Faible', 'Prédit : Élevé'],
            yticklabels=['Réel : Faible', 'Réel : Élevé'],
            ax=ax, cbar=False,
            annot_kws={'size': 14, 'weight': 'bold'})
ax.set_title('Matrice de Confusion — XGBoost (Risque Diabète)', fontsize=13,
             fontweight='bold', color=BIOMEX_COLORS['navy'])
plt.tight_layout()
plt.savefig('12_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Génération du Rapport Patient <a id='10'></a>

In [ ]:

# ══════════════════════════════════════════════════════════════════════
# GÉNÉRATION DU RAPPORT PATIENT COMPLET
# ══════════════════════════════════════════════════════════════════════

def generate_patient_report(patient_id: str, report: dict,
                             taxa_vector: pd.Series,
                             df_scores: pd.DataFrame,
                             save_path: str = None) -> str:
    """
    Génère un rapport patient complet multi-panneaux.
    Production : exporter en PDF via ReportLab.
    """
    fig = plt.figure(figsize=(20, 24))
    fig.patch.set_facecolor('#F7FBFF')

    gs = fig.add_gridspec(4, 3, hspace=0.45, wspace=0.35,
                          top=0.93, bottom=0.05, left=0.06, right=0.97)

    # ── Header ────────────────────────────────────────────────────────
    ax_header = fig.add_subplot(gs[0, :])
    ax_header.set_facecolor(BIOMEX_COLORS['navy'])
    ax_header.text(0.02, 0.70, 'BiomeX', fontsize=32, fontweight='bold',
                   color='#66CCFF', transform=ax_header.transAxes,
                   fontfamily='monospace')
    ax_header.text(0.02, 0.25, f'Rapport Microbiome — {patient_id}  |  Dakar, Sénégal',
                   fontsize=13, color='white', transform=ax_header.transAxes)
    ax_header.text(0.75, 0.50, 'CONFIDENTIEL — Usage médical exclusif',
                   fontsize=11, color='#AACCEE', transform=ax_header.transAxes,
                   style='italic')
    ax_header.set_xlim(0, 1)
    ax_header.set_ylim(0, 1)
    ax_header.axis('off')

    # ── Panel 1 : Gauge Score Global ──────────────────────────────────
    ax1 = fig.add_subplot(gs[1, 0])
    score_global = df_scores.loc[patient_id, 'score_global']
    color_global = (BIOMEX_COLORS['green'] if score_global > 66 else
                    BIOMEX_COLORS['orange'] if score_global > 33 else '#CC0000')

    theta = np.linspace(np.pi, 0, 100)
    ax1.plot(np.cos(theta), np.sin(theta), color='#E0E0E0', linewidth=18)
    ratio = score_global / 100
    theta_fill = np.linspace(np.pi, np.pi - ratio * np.pi, 100)
    ax1.plot(np.cos(theta_fill), np.sin(theta_fill), color=color_global, linewidth=18)
    ax1.text(0, 0.15, f'{score_global:.0f}', ha='center', va='center',
             fontsize=36, fontweight='bold', color=color_global)
    ax1.text(0, -0.25, 'Score Santé Global / 100', ha='center',
             fontsize=10, color='#444444')
    ax1.set_xlim(-1.3, 1.3)
    ax1.set_ylim(-0.5, 1.2)
    ax1.axis('off')
    ax1.set_title('Score Métabolique Global', fontweight='bold',
                  color=BIOMEX_COLORS['navy'], pad=10)

    # ── Panel 2 : Scores radar ────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, 1], projection='polar')
    score_row = df_scores.loc[patient_id]
    cat   = ['Diversité\nShannon', 'Anti-inflam.', 'SCFA', 'Équilibre\n(1-Dysbiose)', 'Global']
    H_n   = min(score_row['shannon_H'] / 4.5, 1.0)
    inf_n = max(0, 1 - (score_row['score_inflam'] + 3) / 6)
    scf_n = min(score_row['score_scfa'] * 10, 1.0)
    dys_n = max(0, 1 - score_row['score_dysbiose'] / 3)
    glo_n = score_row['score_global'] / 100
    vals  = [H_n, inf_n, scf_n, dys_n, glo_n]

    N     = len(cat)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    vals  += vals[:1]

    ax2.set_theta_offset(np.pi / 2)
    ax2.set_theta_direction(-1)
    ax2.plot(angles, vals, color=BIOMEX_COLORS['green'], linewidth=2)
    ax2.fill(angles, vals, alpha=0.25, color=BIOMEX_COLORS['green'])
    ax2.set_xticks(angles[:-1])
    ax2.set_xticklabels(cat, size=9)
    ax2.set_ylim(0, 1)
    ax2.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax2.set_yticklabels(['', '', '', ''], size=7)
    ax2.set_title('Profil Microbiomique', fontweight='bold',
                  color=BIOMEX_COLORS['navy'], pad=20)

    # ── Panel 3 : Top bactéries ───────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 2])
    top_bacteria = taxa_vector.nlargest(10)
    colors_bact  = [BIOMEX_COLORS['orange'] if b in PATHOBIONTES
                    else BIOMEX_COLORS['green'] if b in BUTYRATE_PRODUCERS
                    else BIOMEX_COLORS['blue']
                    for b in top_bacteria.index]
    bars = ax3.barh(range(len(top_bacteria)), top_bacteria.values * 100,
                    color=colors_bact, edgecolor='white')
    ax3.set_yticks(range(len(top_bacteria)))
    ax3.set_yticklabels([b.replace('_', ' ')[:25] for b in top_bacteria.index], fontsize=9)
    ax3.invert_yaxis()
    ax3.set_xlabel('Abondance relative (%)')
    ax3.set_title('Top 10 Espèces Bactériennes', fontweight='bold',
                  color=BIOMEX_COLORS['navy'])

    patches = [
        mpatches.Patch(color=BIOMEX_COLORS['orange'], label='Pathobionte'),
        mpatches.Patch(color=BIOMEX_COLORS['green'],  label='Bénéfique (SCFA)'),
        mpatches.Patch(color=BIOMEX_COLORS['blue'],   label='Neutre'),
    ]
    ax3.legend(handles=patches, loc='lower right', fontsize=8)

    # ── Panel 4 : Recommandations alimentaires ────────────────────────
    ax4 = fig.add_subplot(gs[2, :])
    ax4.set_facecolor(BIOMEX_COLORS['light'])
    ax4.set_xlim(0, 1)
    ax4.set_ylim(0, 1)
    ax4.axis('off')
    ax4.set_title('Recommandations Nutritionnelles Personnalisées', fontweight='bold',
                  color=BIOMEX_COLORS['navy'], fontsize=13, pad=10)

    recs_alim   = report.get('🥗 Recommandations Nutritionnelles', [])
    recs_probio = report.get('🦠 Recommandations Probiotiques', [])
    recs_vie    = report.get('🌙 Recommandations Mode de Vie', [])

    col_width = 0.33
    for col_i, (title_txt, recs, col_color) in enumerate([
        ('🥗 Alimentation',  recs_alim[:4],   BIOMEX_COLORS['green']),
        ('🦠 Probiotiques',  recs_probio[:4], BIOMEX_COLORS['blue']),
        ('🌙 Mode de Vie',   recs_vie[:4],    BIOMEX_COLORS['teal']),
    ]):
        x_start = col_i * col_width + 0.01
        ax4.text(x_start + 0.01, 0.90, title_txt, fontsize=11,
                 fontweight='bold', color=col_color, transform=ax4.transAxes)
        for j, rec in enumerate(recs):
            ax4.text(x_start + 0.01, 0.76 - j * 0.18,
                     f"• {rec[:65]}{'...' if len(rec)>65 else ''}",
                     fontsize=8.5, color='#333333', transform=ax4.transAxes,
                     wrap=True)

    # ── Panel 5 : Alertes + signature ────────────────────────────────
    ax5 = fig.add_subplot(gs[3, :])
    ax5.set_facecolor('#FFF3E0')
    ax5.set_xlim(0, 1)
    ax5.set_ylim(0, 1)
    ax5.axis('off')

    alertes = report.get('⚠️ Alertes Cliniques', ['✅ Aucune alerte'])
    alert_str = '  |  '.join(alertes)
    ax5.text(0.5, 0.65, f'⚠️ Alertes : {alert_str}',
             ha='center', fontsize=10, color='#CC0000' if '🔴' in alert_str else '#555555',
             fontweight='bold', transform=ax5.transAxes)
    ax5.text(0.5, 0.25,
             '⚠️ Ce rapport est un outil d'aide à la décision. Il ne remplace pas l'avis d'un médecin.  '
             '|  BiomeX © 2026  |  abdou.ba@biomex.ai  |  www.biomex.ai',
             ha='center', fontsize=9, color='#888888', style='italic',
             transform=ax5.transAxes)

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight',
                    facecolor=fig.get_facecolor())
        print(f"💾 Rapport sauvegardé : {save_path}")

    plt.show()
    return save_path


# ── Génération du rapport pour le patient sélectionné ─────────────────
report_path = f"BiomeX_Rapport_{patient_id}.png"
generate_patient_report(
    patient_id = patient_id,
    report     = report,
    taxa_vector= taxa_new,
    df_scores  = df_scores,
    save_path  = report_path
)


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SAUVEGARDE DES MODÈLES (Production MLflow)
# ══════════════════════════════════════════════════════════════════════

import pickle, os

os.makedirs('models', exist_ok=True)

# Sauvegarder les modèles
with open('models/rf_model.pkl',  'wb') as f: pickle.dump(rf_model,  f)
with open('models/xgb_model.pkl', 'wb') as f: pickle.dump(xgb_model, f)
with open('models/lr_model.pkl',  'wb') as f: pickle.dump(lr_model,  f)
with open('models/pca.pkl',       'wb') as f: pickle.dump(pca,       f)
with open('models/engine.pkl',    'wb') as f: pickle.dump(engine,    f)

# Sauvegarder les scores de la cohorte
df_scores.to_csv('cohorte_scores_pilote.csv')

print("✅ Modèles sauvegardés dans /models/")
print("   - rf_model.pkl")
print("   - xgb_model.pkl")
print("   - lr_model.pkl")
print("   - pca.pkl")
print("   - engine.pkl")
print("\n✅ Scores cohorte : cohorte_scores_pilote.csv")

print("\n" + "═"*60)
print("  ✅ PIPELINE BIOMEX COMPLET — EXÉCUTION RÉUSSIE")
print("═"*60)
print()
print("  Étapes complétées :")
steps = [
    ("FastQC + Trimmomatic + MEGAHIT",   "Contrôle qualité & assemblage"),
    ("Kraken2 + HUMAnN3",               "Annotation taxo. & fonctionnelle"),
    ("Normalisation CLR + Matrice",      f"X_total : {X_total.shape}"),
    ("PCA + Autoencoder + RF Select",    f"{n_components_90} composantes retenues"),
    ("5 Scores biologiques",             f"Shannon H, Inflam, SCFA, Dysbiose, Global"),
    ("Random Forest",                    f"AUC = {evaluate_model(rf_model,  X_test, y_test, '')['AUC']:.3f}"),
    ("XGBoost",                          f"AUC = {evaluate_model(xgb_model, X_test, y_test, '')['AUC']:.3f}"),
    ("Logistic Regression",              f"AUC = {evaluate_model(lr_model,  X_test, y_test, '')['AUC']:.3f}"),
    ("SHAP Interpretability",            "Explicabilité clinique validée"),
    ("Rapport Patient",                  "Généré — prêt pour intégration app"),
]
for step, detail in steps:
    print(f"  ✅ {step:<35} → {detail}")
print()